In [1]:
# ============================================================
# PASCAL VOC XML -> YOLO CONVERTER
# ============================================================

from pathlib import Path
import xml.etree.ElementTree as ET
import shutil
import yaml
import pandas as pd


# ============================================================
# 1. CONFIGURATION
# ============================================================

# Исходный датасет
DATASET_PATH = Path(
    r"dataset_4"
)

# Куда сохранить YOLO-версию
OUTPUT_PATH = DATASET_PATH.parent / f"{DATASET_PATH.name}_yolo"


# Классы датасета
CLASS_NAMES = {
    0: "tractor",
    1: "truck",
}


# ============================================================
# 2. OUTPUT DIRECTORIES
# ============================================================

IMAGES_OUT = OUTPUT_PATH / "images"
LABELS_OUT = OUTPUT_PATH / "labels"

IMAGES_OUT.mkdir(
    parents=True,
    exist_ok=True
)

LABELS_OUT.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 70)
print("PASCAL VOC XML -> YOLO")
print("=" * 70)

print(f"Source: {DATASET_PATH}")
print(f"Output: {OUTPUT_PATH}")
print()


# ============================================================
# 3. FIND XML FILES
# ============================================================

# В твоей структуре:
#
# dataset_4/
# ├── Annotations/
# │   └── Annotations/
# │       ├── tractor/
# │       └── truck/
# ├── Tractor/
# └── Truck/
#
# Поэтому ищем XML рекурсивно по всему датасету.

xml_files = sorted(
    DATASET_PATH.rglob("*.xml")
)

print(f"XML files found: {len(xml_files)}")
print()


if len(xml_files) == 0:
    raise FileNotFoundError(
        "Не найдено XML-файлов."
    )


# ============================================================
# 4. STATISTICS
# ============================================================

class_counts = {
    class_id: 0
    for class_id in CLASS_NAMES
}

image_counts = {
    class_id: set()
    for class_id in CLASS_NAMES
}

unknown_classes = set()

converted_files = []
missing_images = []
invalid_xml = []


# ============================================================
# 5. HELPER FUNCTIONS
# ============================================================

def find_image(xml_path, filename):
    """
    Ищет соответствующее изображение.

    Сначала пытается найти файл по имени рядом с XML,
    затем рекурсивно внутри dataset_4.
    """

    # --------------------------------------------------------
    # 1. Рядом с XML
    # --------------------------------------------------------

    candidate = xml_path.parent / filename

    if candidate.exists():
        return candidate

    # --------------------------------------------------------
    # 2. По всему датасету
    # --------------------------------------------------------

    matches = list(
        DATASET_PATH.rglob(filename)
    )

    if matches:
        return matches[0]

    return None


def convert_bbox_to_yolo(
    xmin,
    ymin,
    xmax,
    ymax,
    image_width,
    image_height
):
    """
    Pascal VOC bbox -> YOLO bbox
    """

    # Защита от некорректных координат
    xmin = max(0, min(xmin, image_width))
    xmax = max(0, min(xmax, image_width))

    ymin = max(0, min(ymin, image_height))
    ymax = max(0, min(ymax, image_height))

    # Если координаты перепутаны
    if xmax < xmin:
        xmin, xmax = xmax, xmin

    if ymax < ymin:
        ymin, ymax = ymax, ymin

    # Центр
    x_center = (
        (xmin + xmax) / 2
    ) / image_width

    y_center = (
        (ymin + ymax) / 2
    ) / image_height

    # Размер
    width = (
        xmax - xmin
    ) / image_width

    height = (
        ymax - ymin
    ) / image_height

    return (
        x_center,
        y_center,
        width,
        height
    )


# ============================================================
# 6. PROCESS XML FILES
# ============================================================

for xml_path in xml_files:

    try:

        tree = ET.parse(xml_path)
        root = tree.getroot()

    except Exception as e:

        invalid_xml.append({
            "xml": str(xml_path),
            "error": str(e)
        })

        continue


    # ========================================================
    # IMAGE FILENAME
    # ========================================================

    filename_node = root.find("filename")

    if filename_node is None or not filename_node.text:

        invalid_xml.append({
            "xml": str(xml_path),
            "error": "Missing filename"
        })

        continue


    filename = filename_node.text.strip()


    # ========================================================
    # IMAGE SIZE
    # ========================================================

    size_node = root.find("size")

    if size_node is None:

        invalid_xml.append({
            "xml": str(xml_path),
            "error": "Missing size"
        })

        continue


    try:

        image_width = float(
            size_node.find("width").text
        )

        image_height = float(
            size_node.find("height").text
        )

    except Exception:

        invalid_xml.append({
            "xml": str(xml_path),
            "error": "Invalid image dimensions"
        })

        continue


    if image_width <= 0 or image_height <= 0:

        invalid_xml.append({
            "xml": str(xml_path),
            "error": "Image dimensions <= 0"
        })

        continue


    # ========================================================
    # FIND IMAGE
    # ========================================================

    image_path = find_image(
        xml_path,
        filename
    )

    if image_path is None:

        missing_images.append({
            "xml": str(xml_path),
            "filename": filename
        })

        continue


    # ========================================================
    # DETERMINE CATEGORY
    # ========================================================

    # Используем имя подпапки:
    #
    # Annotations/Annotations/tractor/
    # Annotations/Annotations/truck/

    category = xml_path.parent.name.lower()


    # ========================================================
    # OUTPUT IMAGE NAME
    # ========================================================

    # Чтобы избежать конфликтов имен,
    # используем имя категории.
    #
    # tractor_20210530_06_07_18.jpg
    # truck_20210530_06_07_18.jpg

    original_stem = image_path.stem
    extension = image_path.suffix

    output_stem = (
        f"{category}_{original_stem}"
    )

    output_image_path = (
        IMAGES_OUT /
        f"{output_stem}{extension}"
    )

    output_label_path = (
        LABELS_OUT /
        f"{output_stem}.txt"
    )


    # ========================================================
    # PARSE OBJECTS
    # ========================================================

    yolo_lines = []


    for obj in root.findall("object"):

        name_node = obj.find("name")

        if name_node is None or not name_node.text:
            continue

        class_name = (
            name_node.text
            .strip()
            .lower()
        )


        # ----------------------------------------------------
        # FIND CLASS ID
        # ----------------------------------------------------

        class_id = None

        for cid, cname in CLASS_NAMES.items():

            if cname.lower() == class_name:
                class_id = cid
                break


        # ----------------------------------------------------
        # UNKNOWN CLASS
        # ----------------------------------------------------

        if class_id is None:

            unknown_classes.add(class_name)

            continue


        # ----------------------------------------------------
        # BBOX
        # ----------------------------------------------------

        bbox = obj.find("bndbox")

        if bbox is None:
            continue


        try:

            xmin = float(
                bbox.find("xmin").text
            )

            ymin = float(
                bbox.find("ymin").text
            )

            xmax = float(
                bbox.find("xmax").text
            )

            ymax = float(
                bbox.find("ymax").text
            )

        except Exception:

            continue


        # ----------------------------------------------------
        # CONVERT
        # ----------------------------------------------------

        (
            x_center,
            y_center,
            box_width,
            box_height
        ) = convert_bbox_to_yolo(
            xmin,
            ymin,
            xmax,
            ymax,
            image_width,
            image_height
        )


        # ----------------------------------------------------
        # IGNORE INVALID BOXES
        # ----------------------------------------------------

        if box_width <= 0 or box_height <= 0:
            continue


        # ----------------------------------------------------
        # YOLO LINE
        # ----------------------------------------------------

        yolo_line = (
            f"{class_id} "
            f"{x_center:.6f} "
            f"{y_center:.6f} "
            f"{box_width:.6f} "
            f"{box_height:.6f}"
        )

        yolo_lines.append(
            yolo_line
        )


        # ----------------------------------------------------
        # STATISTICS
        # ----------------------------------------------------

        class_counts[class_id] += 1

        image_counts[class_id].add(
            output_stem
        )


    # ========================================================
    # SAVE IMAGE + LABEL
    # ========================================================

    if yolo_lines:

        shutil.copy2(
            image_path,
            output_image_path
        )

        with open(
            output_label_path,
            "w",
            encoding="utf-8"
        ) as f:

            f.write(
                "\n".join(yolo_lines)
            )

        converted_files.append({
            "xml": str(xml_path),
            "image": str(output_image_path),
            "label": str(output_label_path),
            "objects": len(yolo_lines)
        })


# ============================================================
# 7. CREATE YAML
# ============================================================

yaml_data = {
    "path": str(
        OUTPUT_PATH.resolve()
    ),

    "train": "images",

    "val": "images",

    "nc": len(CLASS_NAMES),

    "names": [
        CLASS_NAMES[i]
        for i in range(len(CLASS_NAMES))
    ]
}


yaml_path = OUTPUT_PATH / "dataset.yaml"


with open(
    yaml_path,
    "w",
    encoding="utf-8"
) as f:

    yaml.dump(
        yaml_data,
        f,
        allow_unicode=True,
        sort_keys=False
    )


# ============================================================
# 8. SAVE CONVERSION STATISTICS
# ============================================================

stats_rows = []

total_objects = sum(
    class_counts.values()
)

total_images = len(converted_files)


for class_id, class_name in CLASS_NAMES.items():

    objects = class_counts[class_id]

    images = len(
        image_counts[class_id]
    )

    object_percentage = (
        objects / total_objects * 100
        if total_objects > 0
        else 0
    )

    image_percentage = (
        images / total_images * 100
        if total_images > 0
        else 0
    )

    stats_rows.append({
        "class_id": class_id,
        "class_name": class_name,
        "objects": objects,
        "object_percentage": object_percentage,
        "images": images,
        "image_percentage": image_percentage,
    })


stats_df = pd.DataFrame(
    stats_rows
)


stats_df.to_csv(
    OUTPUT_PATH / "conversion_class_statistics.csv",
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 9. SAVE CONVERSION REPORT
# ============================================================

report = []

report.append(
    "PASCAL VOC -> YOLO CONVERSION REPORT"
)

report.append("=" * 60)

report.append(
    f"Source dataset: {DATASET_PATH}"
)

report.append(
    f"Output dataset: {OUTPUT_PATH}"
)

report.append("")

report.append(
    f"XML files found: {len(xml_files)}"
)

report.append(
    f"Images converted: {total_images}"
)

report.append(
    f"Total objects: {total_objects}"
)

report.append(
    f"Missing images: {len(missing_images)}"
)

report.append(
    f"Invalid XML files: {len(invalid_xml)}"
)

report.append(
    f"Unknown classes: {len(unknown_classes)}"
)

report.append("")

report.append(
    "CLASS DISTRIBUTION"
)

report.append("-" * 60)


for _, row in stats_df.iterrows():

    report.append(
        f"{int(row['class_id'])} | "
        f"{row['class_name']} | "
        f"objects: {int(row['objects'])} | "
        f"images: {int(row['images'])}"
    )


with open(
    OUTPUT_PATH / "conversion_report.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "\n".join(report)
    )


# ============================================================
# 10. SAVE PROBLEM REPORTS
# ============================================================

if missing_images:

    pd.DataFrame(
        missing_images
    ).to_csv(
        OUTPUT_PATH / "missing_images.csv",
        index=False,
        encoding="utf-8-sig"
    )


if invalid_xml:

    pd.DataFrame(
        invalid_xml
    ).to_csv(
        OUTPUT_PATH / "invalid_xml.csv",
        index=False,
        encoding="utf-8-sig"
    )


if unknown_classes:

    pd.DataFrame({
        "unknown_class": sorted(
            unknown_classes
        )
    }).to_csv(
        OUTPUT_PATH / "unknown_classes.csv",
        index=False,
        encoding="utf-8-sig"
    )


# ============================================================
# 11. FINAL OUTPUT
# ============================================================

print()
print("=" * 70)
print("CONVERSION FINISHED")
print("=" * 70)

print()
print(f"XML files:       {len(xml_files)}")
print(f"Images converted:{total_images}")
print(f"Objects:         {total_objects}")
print(
    f"Missing images:  {len(missing_images)}"
)
print(
    f"Invalid XML:     {len(invalid_xml)}"
)
print(
    f"Unknown classes: {len(unknown_classes)}"
)

print()
print("Class distribution:")
print(stats_df.to_string(index=False))

print()
print("Output:")
print(OUTPUT_PATH.resolve())

print()
print("YAML:")
print(yaml_path.resolve())

PASCAL VOC XML -> YOLO
Source: dataset_4
Output: dataset_4_yolo

XML files found: 326


CONVERSION FINISHED

XML files:       326
Images converted:319
Objects:         427
Missing images:  0
Invalid XML:     0
Unknown classes: 0

Class distribution:
 class_id class_name  objects  object_percentage  images  image_percentage
        0    tractor      119          27.868852     107         33.542320
        1      truck      308          72.131148     218         68.338558

Output:
C:\Users\Ева\Desktop\Хакатон\data_preproc\dataset_4_yolo

YAML:
C:\Users\Ева\Desktop\Хакатон\data_preproc\dataset_4_yolo\dataset.yaml
